In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")
dbutils.widgets.text("load_type", "FULL")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")
load_type = dbutils.widgets.get("load_type")
normalized_load_type = (load_type or "FULL").strip().upper()


In [0]:
base_landing_path = "/Volumes/adb_caresync/landing/landing_volume"
source_path = f"{base_landing_path}/{source_system}"
table_path = f"{source_path}/{table_name}"
landing_path = f"{table_path}/{landing_timestamp}/"

# Parent paths must exist and be accessible. Do not mask permission or storage errors.
dbutils.fs.ls(source_path)
dbutils.fs.ls(table_path)

try:
    landing_entries = dbutils.fs.ls(landing_path)
except Exception as exc:
    error_text = str(exc).lower()
    missing_path = any(token in error_text for token in (
        "filenotfoundexception", "pathnotfound", "path does not exist"
    ))
    if not missing_path:
        raise
    landing_entries = []

landing_files = [entry for entry in landing_entries if not entry.isDir()]

def stop_for_no_data(reason: str) -> None:
    message = (
        f"{reason} source_system={source_system}, table_name={table_name}, "
        f"load_type={normalized_load_type}, landing_timestamp={landing_timestamp}."
    )
    if normalized_load_type == "FULL":
        raise RuntimeError(message + " FULL loads cannot proceed without data.")
    print(message + " Returning NO_DATA; Silver and watermark updates must be skipped.")
    dbutils.notebook.exit("NO_DATA")

if not landing_files:
    stop_for_no_data("No landing files found for this batch.")


In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

# Treat an empty extract as NO_DATA for incremental loads; never overwrite FULL Silver with zero rows.
if landing_df.limit(1).count() == 0:
    stop_for_no_data("Landing batch contains zero records.")


In [0]:
from pyspark.sql import functions as F

updated_df = landing_df.withColumn("landing_timestamp", F.lit(landing_timestamp)) \
    .withColumn("insert_timestamp", F.current_timestamp())


In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")